[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_07/MFM_ch7_seminar/MFM_ch7_seminar.ipynb)

# MFM_ch7_seminar

Seminar 7 computations: Normal and Student-t VaR/ES step by step, discrete losses and the failure of subadditivity, component VaR, Cornish-Fisher, historical VaR/ES on BET with bootstrap intervals, GPD fit and EVT-VaR on BET, filtered vs historical simulation on Bitcoin, Cornish-Fisher vs empirical quantiles, Euler contributions for a four-asset portfolio, i.i.d. and block bootstrap intervals for ES, asymptotic (i.i.d. and HAC) intervals for historical VaR, the extremal index of BET losses, a residual-bootstrap interval for the filtered VaR of Bitcoin, CAViaR on BET, and the ES 2.5% capital of a BVB blue-chip portfolio versus an S&P 500 portfolio.

*Modelling Financial Markets (MFM), Chapter 7: Value-at-Risk and Expected Shortfall. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_7'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install arch
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate, signal
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Brand colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'   # reference lines, bands and grid only
LightGray = '#DADADA'
Teal     = '#17A2B8'
METHOD_COL = {'HS': MainBlue, 'Normal': Orange, 'Student-t': Forest, 'Cornish-Fisher': Purple, 'EVT (GPD)': IDAred}

HERE = '.'
SEED = 42
ORDER = ['sp500', 'bet', 'btc', 'eurron', 'gold']
METHODS = ['HS', 'Normal', 'Student-t', 'Cornish-Fisher', 'EVT (GPD)']
B_BOOT = 2000


def save_fig(name):
    """Save the figure as a transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# use the local copy of the course data when the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, type, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',    'S&P 500',                  'index',  '2000-01-01'),
    'bet':    ('BET',          'BET',                      'index',  '2000-01-01'),
    'btc':    ('BTC-USD.CC',   'Bitcoin',                  'crypto', '2014-09-17'),
    'eurron': ('REF:EUR',      'EUR/RON (reference rate)', 'fx',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX', 'Gold (XAU/USD)',           'fx',     '2000-01-01'),
}
# portfolio assets (prices aligned on common days)
ASSETS = {
    'SPY': ('SPY.US', 'adjusted_close'), 'TLT': ('TLT.US', 'adjusted_close'), 'GLD': ('GLD.US', 'adjusted_close'),
    'BTC': ('BTC-USD.CC', 'close'),
    'TLV': ('TLV.RO', 'adjusted_close'), 'SNP': ('SNP.RO', 'adjusted_close'), 'BRD': ('BRD.RO', 'adjusted_close'),
    'TGN': ('TGN.RO', 'adjusted_close'), 'SNG': ('SNG.RO', 'adjusted_close'), 'SNN': ('SNN.RO', 'adjusted_close'),
    'EL': ('EL.RO', 'adjusted_close'), 'FP': ('FP.RO', 'adjusted_close'), 'TEL': ('TEL.RO', 'adjusted_close'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Read the daily price series of one symbol (local copy of the course data, else from GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (annual BNR XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily price, cleaned according to the chapter conventions."""
    symbol, _, kind, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    d = read_market(symbol).loc[start:end]
    d = d[d['close'] > 0].dropna(subset=['close'])
    if kind != 'crypto':
        d = d[d.index.dayofweek < 5]          # no weekend quotes
    s = d['close']
    if kind == 'index' and 'volume' in d:
        # drop holidays filled with the previous price: unchanged close and zero/missing volume
        s = s[~((s.diff() == 0) & (d['volume'].fillna(0) <= 0))]
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def periods_per_year(r):
    """Actual frequency: average number of observations per calendar year."""
    return len(r) / ((r.index[-1] - r.index[0]).days / 365.25)


def asset_price(key, end=END):
    symbol, col = ASSETS[key]
    s = read_market(symbol)[col].loc[:end]
    s = s[s > 0].dropna()
    if key != 'BTC':
        s = s[s.index.dayofweek < 5]
    return s.rename(key)


def joint_returns(keys, start=None, end=END):
    """Log returns for several assets: align the PRICES on common days first, then compute returns."""
    p = pd.concat([asset_price(k, end) for k in keys], axis=1, join='inner').dropna()
    if start:
        p = p.loc[start:]
    return np.log(p).diff().dropna()


# daily log returns in %, loss L = -r
rets = {k: 100 * log_returns(k) for k in MARKETS}
loss = {k: -v for k, v in rets.items()}
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

## Risk measures

In [ ]:
def hs_var_es(L, alpha):
    """Historical simulation: VaR alpha = empirical (1 - alpha)-quantile of the loss; ES = mean of the losses beyond it."""
    L = np.asarray(L)
    v = np.quantile(L, 1 - alpha)
    return v, L[L >= v].mean()


def normal_var_es(L, alpha):
    """Distributia Normala: VaR = -mu_X + sigma z_{1-alpha}, ES = -mu_X + sigma phi(z_alpha)/alpha (mu_L = -mu_X)."""
    mu, s = np.mean(L), np.std(L, ddof=1)
    z = stats.norm.ppf(1 - alpha)
    return mu + s * z, mu + s * stats.norm.pdf(z) / alpha


def t_fit(L):
    """Student-t with location and scale, fitted by maximum likelihood."""
    return stats.t.fit(np.asarray(L))


def t_var_es(L, alpha, params=None):
    """Student-t on losses: VaR = m + s t_{1-alpha}, ES = m + s g(t_alpha)/alpha (nu + t_alpha^2)/(nu - 1)."""
    nu, m, s = params if params is not None else t_fit(L)
    q = stats.t.ppf(1 - alpha, nu)
    return m + s * q, m + s * stats.t.pdf(q, nu) / alpha * (nu + q ** 2) / (nu - 1)


def cf_quantile(z, S, K):
    """Standardised Cornish-Fisher quantile: z = Normal quantile, S = skewness, K = excess kurtosis."""
    return z + (z ** 2 - 1) * S / 6 + (z ** 3 - 3 * z) * K / 24 - (2 * z ** 3 - 5 * z) * S ** 2 / 36


def cf_var_es(L, alpha):
    """Cornish-Fisher on returns X = -L: VaR = -(mu_X + sigma z~_alpha); ES = mean of VaR_u for u in (0, alpha)."""
    X = -np.asarray(L)
    mu, s = X.mean(), X.std(ddof=1)
    S, K = stats.skew(X), stats.kurtosis(X)
    var = -(mu + s * cf_quantile(stats.norm.ppf(alpha), S, K))
    es = -integrate.quad(lambda u: mu + s * cf_quantile(stats.norm.ppf(u), S, K), 0, alpha, limit=200)[0] / alpha
    return var, es


def gpd_fit(L, tail=0.05):
    """POT: the threshold u leaves the share tail of the losses above it; GPD fitted by maximum likelihood to the excesses."""
    L = np.asarray(L)
    u = np.quantile(L, 1 - tail)
    ex = L[L > u] - u
    xi, _, beta = stats.genpareto.fit(ex, floc=0)
    return dict(u=u, xi=xi, beta=beta, nu=len(ex), n=len(L))


def gpd_var_es(fit, alpha):
    """VaR and ES at tail probability alpha from the GPD tail (Smith-type estimator)."""
    u, xi, beta, nu, n = fit['u'], fit['xi'], fit['beta'], fit['nu'], fit['n']
    var = u + beta / xi * (((n / nu) * alpha) ** (-xi) - 1)
    return var, var / (1 - xi) + (beta - xi * u) / (1 - xi)


def gpd_se(fit):
    """Asymptotic standard errors of the GPD maximum likelihood estimates (xi > -1/2)."""
    xi, beta, nu = fit['xi'], fit['beta'], fit['nu']
    return np.sqrt((1 + xi) ** 2 / nu), np.sqrt(2 * beta ** 2 * (1 + xi) / nu)


def mean_excess(L, us):
    """Mean excess function e(u) = E[L - u | L > u] on a grid of thresholds."""
    L = np.asarray(L)
    return np.array([(L[L > u] - u).mean() for u in us]), np.array([(L > u).sum() for u in us])


def garch_backcast(x):
    """Starting value of the variance (as in arch): EWMA mean (0.94) of the first 75 squared residuals
    of an AR(1) fitted by OLS, computed ONLY on the estimation sample x."""
    x = np.asarray(x, float)
    X = np.column_stack([np.ones(len(x) - 1), x[:-1]])
    e = x[1:] - X @ np.linalg.lstsq(X, x[1:], rcond=None)[0]
    tau = min(75, len(e))
    w = 0.94 ** np.arange(tau)
    return float((w / w.sum()) @ e[:tau] ** 2)


def garch_filter(r, params=None, last_obs=None):
    """AR(1)-GARCH(1,1) by Normal QML (Chapter 5); r in %.
    Parameters are estimated on the data before last_obs; the filter is causal: mu_t and sigma_t use
    only r_1..r_{t-1}, and the starting variance comes only from the estimation sample.
    Returns the parameters and the series mu_t, sigma_t for the whole sample."""
    if params is None:
        am = arch_model(r, mean='AR', lags=1, vol='GARCH', p=1, q=1, dist='normal', rescale=False)
        params = am.fit(disp='off', last_obs=last_obs).params
    est = r if last_obs is None else r.loc[r.index < pd.Timestamp(last_obs)]
    c, phi, om, al, be = params.values[:5]
    x = np.asarray(r, float)
    mu = np.full(len(x), np.nan)
    mu[1:] = c + phi * x[:-1]
    e = x - mu
    u = np.empty(len(x) - 1)
    u[0] = om + (al + be) * garch_backcast(np.asarray(est, float))
    u[1:] = om + al * e[1:-1] ** 2
    s2 = np.full(len(x), np.nan)
    s2[1:] = signal.lfilter([1.0], [1.0, -be], u)
    return params, pd.Series(mu, index=r.index), pd.Series(np.sqrt(s2), index=r.index)


def garch_next(r, params):
    """Conditional mean and volatility for the day after the last observation."""
    c, phi, om, al, be = params.values[:5]
    _, mu, sig = garch_filter(r, params)
    e = r.iloc[-1] - mu.iloc[-1]
    return c + phi * r.iloc[-1], np.sqrt(om + al * e ** 2 + be * sig.iloc[-1] ** 2)


def fhs_mc(r, params, z, h, n_paths=100_000, seed=42):
    """FHS Monte Carlo: h-day AR(1)-GARCH(1,1) paths with resampled standardised residuals.
    Returns the cumulative h-day losses (in %, log returns)."""
    rng = np.random.default_rng(seed)
    c, phi, om, al, be = params.values[:5]
    _, mu, sig = garch_filter(r, params)
    r_prev = np.full(n_paths, r.iloc[-1])
    e_prev = np.full(n_paths, r.iloc[-1] - mu.iloc[-1])
    s2_prev = np.full(n_paths, sig.iloc[-1] ** 2)
    tot = np.zeros(n_paths)
    for _ in range(h):
        s2 = om + al * e_prev ** 2 + be * s2_prev
        e = np.sqrt(s2) * rng.choice(z, n_paths)
        r_new = c + phi * r_prev + e
        tot += r_new
        r_prev, e_prev, s2_prev = r_new, e, s2
    return -tot


def rolling_conditional(r, first_year, window_hs=500, alpha=0.01, tail_evt=0.10):
    """Day-by-day conditional VaR/ES: AR(1)-GARCH parameters re-estimated at the start of every year
    on all earlier data; FHS and conditional EVT use the earlier standardised residuals.
    For comparison: HS on a rolling window of window_hs days. No look-ahead bias."""
    out = []
    L = -r
    for y in range(first_year, r.index[-1].year + 1):
        est = r.loc[:f'{y - 1}-12-31']
        params, mu, sig = garch_filter(r, last_obs=f'{y}-01-01')
        z = ((est - mu.loc[est.index]) / sig.loc[est.index]).dropna()
        nz = -z.values
        fz = gpd_fit(nz, tail_evt)
        q_fhs, q_evt_v = np.quantile(nz, 1 - alpha), gpd_var_es(fz, alpha)[0]
        es_fhs = nz[nz >= q_fhs].mean()
        es_evt = gpd_var_es(fz, alpha)[1]
        days = r.loc[f'{y}-01-01':f'{y}-12-31'].index
        for d in days:
            i = r.index.get_loc(d)
            past = L.iloc[max(0, i - window_hs):i]
            out.append((d, L.loc[d], -mu.loc[d] + sig.loc[d] * q_fhs, -mu.loc[d] + sig.loc[d] * es_fhs,
                        -mu.loc[d] + sig.loc[d] * q_evt_v, -mu.loc[d] + sig.loc[d] * es_evt,
                        -mu.loc[d] + sig.loc[d] * stats.norm.ppf(1 - alpha), np.quantile(past, 1 - alpha), sig.loc[d]))
    return pd.DataFrame(out, columns=['date', 'loss', 'fhs_var', 'fhs_es', 'cevt_var', 'cevt_es', 'ngarch_var',
                                      'hs_var', 'sigma']).set_index('date')

## Helper functions

In [ ]:
from scipy import optimize, signal

BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'SNG', 'SNN', 'EL', 'TEL']

C_START = '2016-09-19'

W_FIG = 5.6

def discrete_var_es(values, probs, alpha):
    """VaR and ES at tail probability alpha for a discrete loss L = -X (Acerbi-Tasche formula).
    VaR_alpha = -inf{x : P(X <= x) > alpha}; ES_alpha = (E[L 1{L > VaR}] + VaR (alpha - P(L > VaR))) / alpha."""
    order = np.argsort(values)
    x, p = np.asarray(values)[order], np.asarray(probs)[order]
    cdf = np.cumsum(p)
    var = x[np.searchsorted(cdf, 1 - alpha - 1e-12)]
    tail = x > var
    es = ((x[tail] * p[tail]).sum() + var * (alpha - p[tail].sum())) / alpha
    return var, es


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o

## Analysis

In [ ]:
def a1_normal(mu=0.04, sigma=1.2, position=1_000_000):
    """VaR and ES for the Normal distribution, step by step: X ~ N(mu, sigma^2) (returns in %),
    VaR_alpha = -(mu + sigma z_alpha), ES_alpha = -mu + sigma phi(z_alpha) / alpha."""
    out = dict(mu=mu, sigma=sigma, position=position)
    for a, tag in [(0.01, '1'), (0.025, '2_5')]:
        z = stats.norm.ppf(a)                  # z_alpha < 0
        phi = stats.norm.pdf(z)
        out[f'z{tag}'] = z
        out[f'phi{tag}'] = phi
        out[f'k{tag}'] = phi / a
        out[f'var{tag}'] = -(mu + sigma * z)
        out[f'es{tag}'] = -mu + sigma * phi / a
    out['var1_10'] = -10 * mu - np.sqrt(10) * sigma * out['z1']   # 10 days, additive i.i.d. returns
    return out


def a2_student(nu=4, mu=0.04, sigma=1.2):
    """Student-t scaled to the same variance: X = mu + s T, s = sigma sqrt((nu-2)/nu);
    VaR_alpha = -(mu + s t_alpha), ES_alpha = -mu + s g(t_alpha)/alpha (nu + t_alpha^2)/(nu - 1)."""
    s = sigma * np.sqrt((nu - 2) / nu)
    out = dict(nu=nu, s=s)
    for a, tag in [(0.01, '1'), (0.025, '2_5')]:
        q = stats.t.ppf(a, nu)                 # t_alpha < 0
        g = stats.t.pdf(q, nu)
        k = g / a * (nu + q ** 2) / (nu - 1)
        out[f'q{tag}'] = q
        out[f'g{tag}'] = g
        out[f'k{tag}'] = k
        out[f'var{tag}'] = -(mu + s * q)
        out[f'es{tag}'] = -mu + s * k
    return out


def a3_bond(p=0.03, lgd=60.0):
    out = dict(p=p, lgd=lgd)
    for a, tag in [(0.05, '5'), (0.025, '2_5')]:
        v, e = discrete_var_es([0, lgd], [1 - p, p], a)
        out[f'var{tag}'] = v
        out[f'es{tag}'] = e
    return out


def a4_bonds(p=0.007, lgd=100.0):
    out = dict(p=p, lgd=lgd, p_any=1 - (1 - p) ** 2, p_both=p ** 2, p_one=2 * p * (1 - p))
    for a, tag in [(0.01, '1'), (0.015, '1_5')]:
        v1, e1 = discrete_var_es([0, lgd], [1 - p, p], a)
        vp, ep = discrete_var_es([0, lgd, 2 * lgd], [(1 - p) ** 2, 2 * p * (1 - p), p ** 2], a)
        out.update({f'varA_{tag}': v1, f'esA_{tag}': e1, f'varP_{tag}': vp, f'esP_{tag}': ep})
    return out


def a5_components(pos=(600_000, 400_000), sig=(1.2, 0.9), rho=-0.2, alpha=0.01):
    """Component VaR 1% for two assets, Normal distribution, zero mean."""
    w = np.array(pos, float)
    s = np.array(sig) / 100
    S = np.array([[s[0] ** 2, rho * s[0] * s[1]], [rho * s[0] * s[1], s[1] ** 2]])
    sp = np.sqrt(w @ S @ w)
    z = stats.norm.ppf(1 - alpha)              # z_{1-alpha} = -z_alpha > 0
    Sw = S @ w
    mvar = z * Sw / sp
    cvar = w * mvar
    return dict(w=w, s=s, rho=rho, z=z, var_p_amt=z * sp, sigma_p_amt=sp, sp2=sp ** 2, Sw=Sw, mvar=mvar, cvar=cvar,
                share=cvar / (z * sp), var_ind=z * s * w, undiv=(z * s * w).sum())


def a6_cf(S=-0.5, K=3.0, sigma=1.2, mu=0.0, alpha=0.01, K2=10.0):
    """Cornish-Fisher on returns: skewness S, excess kurtosis K; VaR_alpha = -(mu + sigma z~_alpha)."""
    z = stats.norm.ppf(alpha)                  # z_alpha < 0
    t1, t2, t3 = (z ** 2 - 1) * S / 6, (z ** 3 - 3 * z) * K / 24, -(2 * z ** 3 - 5 * z) * S ** 2 / 36
    zc = cf_quantile(z, S, K)
    zc2 = cf_quantile(z, S, K2)
    return dict(S=S, K=K, K2=K2, z=z, t1=t1, t2=t2, t3=t3, zcf=zc, var_cf=-(mu + sigma * zc), var_n=-(mu + sigma * z),
                zcf2=zc2, var_cf2=-(mu + sigma * zc2))


def boot_ci(L, fun, B=B_BOOT, block=None, seed=SEED):
    """95% percentile bootstrap interval: i.i.d. (block=None) or moving blocks of length block."""
    rng = np.random.default_rng(seed)
    L = np.asarray(L)
    n = len(L)
    stats_ = np.empty(B)
    for b in range(B):
        if block is None:
            x = L[rng.integers(0, n, n)]
        else:
            starts = rng.integers(0, n - block + 1, int(np.ceil(n / block)))
            x = np.concatenate([L[s:s + block] for s in starts])[:n]
        stats_[b] = fun(x)
    return np.percentile(stats_, [2.5, 97.5]), stats_


def b1_bet_hs():
    L = -100 * log_returns('bet')
    out = {}
    boots = {}
    for tag, x in [('full', L), ('w500', L.iloc[-500:])]:
        v1, e1 = hs_var_es(x, 0.01)
        v2, e2 = hs_var_es(x, 0.025)
        ci, bs = boot_ci(x.values, lambda y: hs_var_es(y, 0.01)[0])
        boots[tag] = bs
        out[tag] = dict(N=len(x), start=str(x.index[0].date()), var1=v1, es1=e1, var2_5=v2, es2_5=e2,
                        ci_lo=ci[0], ci_hi=ci[1], n_beyond=int((x >= v1).sum()))
    # chart: loss tail + bootstrap distributions
    _style()
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.6))
    ax = axes[0]
    ax.hist(L[L > 1.5], bins=np.linspace(1.5, 13.5, 61), color=MainBlue, alpha=0.55, label='BET daily losses above 1.5%')
    ax.axvline(out['full']['var1'], color=IDAred, lw=1.3, label=f"VaR 1% = {out['full']['var1']:.2f}%")
    ax.axvline(out['full']['es1'], color=IDAred, ls='--', lw=1.3, label=f"ES 1% = {out['full']['es1']:.2f}%")
    ax.set_xlabel('Daily loss (%)')
    ax.set_ylabel('Number of days')
    ax.set_title('BET, 2000-2026: historical simulation')
    pass
    ax = axes[1]
    ax.hist(boots['full'], bins=40, density=True, color=MainBlue, alpha=0.55, label='Full sample (2000-2026)')
    ax.hist(boots['w500'], bins=40, density=True, color=Amber, alpha=0.55, label='Last 500 days')
    ax.set_xlabel('Bootstrap VaR 1% (%)')
    ax.set_ylabel('Density')
    ax.set_title('Sampling uncertainty of HS VaR 1%')
    bottom_legend(fig, ncol=3, fontsize=8.5)
    plt.tight_layout()
    save_fig('ch7_sem_bet_hs')
    return out


def b2_bet_gpd():
    L = (-100 * log_returns('bet')).values
    f = gpd_fit(L, 0.05)
    se_xi, se_beta = gpd_se(f)
    out = dict(u=f['u'], xi=f['xi'], beta=f['beta'], nu=f['nu'], n=f['n'], se_xi=se_xi, se_beta=se_beta)
    for a, tag in [(0.01, '1'), (0.005, '0_5'), (0.001, '0_1')]:
        v, e = gpd_var_es(f, a)
        hv, he = hs_var_es(L, a)
        out.update({f'evt_var{tag}': v, f'evt_es{tag}': e, f'hs_var{tag}': hv, f'hs_es{tag}': he,
                    f'n_beyond{tag}': int((L >= hv).sum())})
    # sensitivity to the threshold
    qs = np.linspace(0.10, 0.015, 18)          # share of losses above the threshold
    xis, lo, hi, v01 = [], [], [], []
    for q in qs:
        g = gpd_fit(L, q)
        s = gpd_se(g)[0]
        xis.append(g['xi'])
        lo.append(g['xi'] - 1.96 * s)
        hi.append(g['xi'] + 1.96 * s)
        v01.append(gpd_var_es(g, 0.001)[0])
    out['xi_range'] = [float(min(xis)), float(max(xis))]
    out['v0_1_range'] = [float(min(v01)), float(max(v01))]
    _style()
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.6))
    ax = axes[0]
    xs = np.sort(L[L > f['u']])
    emp = (1 - np.arange(len(xs)) / len(xs)) * f['nu'] / f['n']
    ax.loglog(xs, emp, 'o', ms=2.5, color=MainBlue, label='Empirical P(L >= x)')
    xx = np.linspace(f['u'], xs[-1] * 1.3, 200)
    ax.loglog(xx, f['nu'] / f['n'] * stats.genpareto.sf(xx - f['u'], f['xi'], scale=f['beta']), color=IDAred,
              label=f"GPD fit, xi = {f['xi']:.2f}")
    ax.loglog(xx, stats.norm.sf(xx, L.mean(), L.std()), color=Orange, ls='--', label='Normal distribution')
    ax.set_ylim(1e-5, 0.08)
    ax.set_xlabel('Daily loss x (%)')
    ax.set_ylabel('Exceedance probability')
    ax.set_title('BET: largest 5% of daily losses')
    pass
    ax = axes[1]
    ax.fill_between(100 * qs, lo, hi, color=LightGray, label='95% confidence band')
    ax.plot(100 * qs, xis, 'o-', ms=3, color=IDAred, label='Estimated xi')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xlabel('Share of losses above u (%)')
    ax.set_ylabel(r'Shape $\hat\xi$')
    ax.set_title(r'Stability of $\xi$ across thresholds')
    bottom_legend(fig, ncol=3, fontsize=8.5)
    plt.tight_layout()
    save_fig('ch7_sem_bet_gpd')
    return out


def b3_btc_fhs():
    r = 100 * log_returns('btc')
    params, mu, sig = garch_filter(r)
    z = ((r - mu) / sig).dropna().values
    m1, s1 = garch_next(r, params)
    nz = -z
    q = np.quantile(nz, 1 - 0.01)
    L = -r
    v500, e500 = hs_var_es(L.iloc[-500:], 0.01)
    vfull, efull = hs_var_es(L, 0.01)
    roll = pd.read_csv(os.path.join(HERE, 'ch7_rolling_btc.csv'), index_col=0, parse_dates=True)
    rec = roll.loc['2024-09-18':]
    out = dict(mu=m1, sigma=s1, sigma_uncond=r.std(), zq=q, zes=nz[nz >= q].mean(), fhs_var=-m1 + s1 * q,
               fhs_es=-m1 + s1 * nz[nz >= q].mean(), hs500_var=v500, hs500_es=e500, hsfull_var=vfull, hsfull_es=efull,
               n_rec=len(rec), exc_hs_rec=int((rec['loss'] > rec['hs_var']).sum()),
               exc_fhs_rec=int((rec['loss'] > rec['fhs_var']).sum()),
               exc_hs_all=float((roll['loss'] > roll['hs_var']).mean()),
               exc_fhs_all=float((roll['loss'] > roll['fhs_var']).mean()), n_all=len(roll),
               alpha=params.iloc[3], beta=params.iloc[4], sd500=r.iloc[-500:].std(), last_r=r.iloc[-1])
    out.update(fig_btc_fhs3(out))
    return out


def b4_cf_vs_empirical():
    t = pd.read_csv(os.path.join(HERE, 'ch7_var_es_table.csv'), index_col=0)
    out = {}
    for k in t.index:
        L = (-100 * log_returns(k)).values
        ci, _ = boot_ci(L, lambda y: hs_var_es(y, 0.01)[0], B=1000)
        out[k] = dict(skew=-t.loc[k, 'skew'], kurt=t.loc[k, 'kurt'], hs1=t.loc[k, 'HS|VaR 1%'],
                      cf1=t.loc[k, 'Cornish-Fisher|VaR 1%'], n1=t.loc[k, 'Normal|VaR 1%'],
                      hs2_5=t.loc[k, 'HS|VaR 2.5%'], cf2_5=t.loc[k, 'Cornish-Fisher|VaR 2.5%'],
                      n2_5=t.loc[k, 'Normal|VaR 2.5%'], ci_lo=ci[0], ci_hi=ci[1])
    return out


def b5_components(w=(0.25, 0.25, 0.25, 0.25)):
    W = np.array(w)
    lr = joint_returns(['SPY', 'TLT', 'GLD', 'BTC'], start='2014-09-18')
    R = 100 * (np.exp(lr) - 1)
    S = np.cov(R.values.T)
    sp = np.sqrt(W @ S @ W)
    z = stats.norm.ppf(1 - 0.01)
    mvar = z * (S @ W) / sp
    cvar = W * mvar
    Lp = -(R.values @ W)
    v, e = hs_var_es(Lp, 0.025)
    ces = -(R.values[Lp >= v] * W).mean(axis=0)
    return dict(N=len(R), var_p=z * sp, mvar=mvar, cvar=cvar, share=cvar / (z * sp), es_p=e, ces=ces,
                ces_share=ces / e, undiv=(z * np.sqrt(np.diag(S)) * W).sum())


def b6_es_boot():
    out = {}
    for k in ['sp500', 'bet']:
        L = (-100 * log_returns(k)).values
        for tag, x in [('full', L), ('w500', L[-500:])]:
            es = hs_var_es(x, 0.025)[1]
            ci_iid, _ = boot_ci(x, lambda y: hs_var_es(y, 0.025)[1])
            ci_blk, _ = boot_ci(x, lambda y: hs_var_es(y, 0.025)[1], block=20)
            out[f'{k}_{tag}'] = dict(N=len(x), es=es, iid_lo=ci_iid[0], iid_hi=ci_iid[1], blk_lo=ci_blk[0],
                                     blk_hi=ci_blk[1])
    return out


def c1_capital(position=1_000_000, B=B_BOOT):
    """Simplified illustration inspired by FRTB (not the full regulatory capital).
    Daily SIMPLE returns (in %): equally weighted BVB basket rebalanced daily, SPY.
    h-day loss = 100 (1 - prod(1 + R_t)) (compounding), so ES in % of the position converts exactly into money.
    Stress window: the 250 consecutive days with the largest ONE-DAY HISTORICAL ES 2.5%; 10-day stressed ES =
    one-day stressed ES x sqrt(10); 20-day liquidity horizon: x sqrt(20/10) (MAR33). Capital = 1.5 x stressed ES.
    Uncertainty: moving-block bootstrap (20 days), with the stress window held FIXED."""
    lr_b = joint_returns(BVB, start=C_START)
    lr_b = lr_b[(lr_b != 0).any(axis=1)]                # drop the days on which no stock price changed
    start = str(lr_b.index[0].date())
    Rb = 100 * (np.exp(lr_b) - 1)
    Rp_b = Rb.mean(axis=1)                               # simple return of the basket, equal weights, rebalanced daily
    Rp_s = 100 * (np.exp(joint_returns(['SPY']).loc[start:]['SPY']) - 1)
    out = dict(start=start, stocks=BVB)
    series = {'bvb': Rp_b, 'spy': Rp_s}
    es25 = lambda y: hs_var_es(y, 0.025)[1]              # noqa: E731
    for k, R in series.items():
        L = -R
        v1, e1 = hs_var_es(L, 0.025)
        L10 = 100 * (1 - (1 + R / 100).rolling(10).apply(np.prod, raw=True)).dropna()   # compounded 10-day losses
        e10 = hs_var_es(L10, 0.025)[1]
        roll = pd.Series([es25(L.iloc[i - 250:i]) for i in range(250, len(L) + 1)], index=L.index[249:])
        es_stress = roll.max()
        end_s = roll.idxmax()
        i_end = L.index.get_loc(end_s)
        start_s = L.index[i_end - 249]
        Ls = L.iloc[i_end - 249:i_end + 1].values
        # FHS: AR(1)-GARCH(1,1) on the portfolio log returns; each path converted exactly
        r = 100 * np.log(1 + R / 100)
        params, mu, sig = garch_filter(r)
        z = ((r - mu) / sig).dropna().values
        m1, s1 = garch_next(r, params)
        sim_log = fhs_mc(r, params, z, 10, n_paths=100_000, seed=SEED)      # 10-day log losses
        fhs10 = hs_var_es(100 * (1 - np.exp(-sim_log / 100)), 0.025)[1]
        # uncertainty: moving-block bootstrap with 20-day blocks
        ci1, _ = boot_ci(L.values, es25, B=B, block=20)
        ci_s, _ = boot_ci(Ls, es25, B=B, block=20)
        out[k] = dict(N=len(R), sd=R.std(), es1=e1, var1=v1, es10_sqrt=np.sqrt(10) * e1, es10_hs=e10,
                      rho1=float(R.autocorr(1)),
                      es_stress1=es_stress, es_stress10=np.sqrt(10) * es_stress, es_stress20=np.sqrt(20) * es_stress,
                      stress_from=str(start_s.date()), stress_to=str(end_s.date()), fhs10=fhs10, sigma_now=s1,
                      es1_lo=ci1[0], es1_hi=ci1[1], es_stress1_lo=ci_s[0], es_stress1_hi=ci_s[1],
                      cap_hs=np.sqrt(10) * e1 / 100 * position, cap_stress=np.sqrt(10) * es_stress / 100 * position,
                      cap_stress_lh20=np.sqrt(20) * es_stress / 100 * position,
                      ima10=1.5 * np.sqrt(10) * es_stress / 100 * position,
                      ima20=1.5 * np.sqrt(20) * es_stress / 100 * position,
                      cap_fhs=fhs10 / 100 * position, zero_share=float((Rb == 0).mean().mean()) if k == 'bvb' else 0.0)
    # correlation: align the portfolio values (BVB wealth index, SPY price) on common days first, then returns
    W = pd.concat([(1 + Rp_b / 100).cumprod(), (1 + Rp_s / 100).cumprod()], axis=1, join='inner')
    out['corr_bvb_spy'] = float(W.pct_change().dropna().corr().iloc[0, 1])
    # chart
    _style()
    fig, ax = plt.subplots(figsize=(W_FIG, 2.6))
    labs = ['HS,\nsqrt(10) x 1-day', 'HS, 10-day\ncompounded', 'Stressed 250\ndays, sqrt(10)', 'FHS Monte\nCarlo, 10-day']
    keys = ['es10_sqrt', 'es10_hs', 'es_stress10', 'fhs10']
    xx = np.arange(len(labs))
    vb = [out['bvb'][c] for c in keys]
    vs = [out['spy'][c] for c in keys]
    ax.bar(xx - 0.2, vb, 0.38, color=IDAred, label='BVB blue chips, equal weights')
    ax.bar(xx + 0.2, vs, 0.38, color=MainBlue, label='S&P 500 (SPY)')
    for i in range(len(labs)):
        ax.text(xx[i] - 0.2, vb[i] + 0.2, f'{vb[i]:.1f}', ha='center', fontsize=7)
        ax.text(xx[i] + 0.2, vs[i] + 0.2, f'{vs[i]:.1f}', ha='center', fontsize=7)
    ax.set_xticks(xx)
    ax.set_xticklabels(labs, fontsize=8)
    ax.set_ylabel('10-day ES 2.5% (% of position)')
    ax.set_title(f'Capital per unit invested, {start[:4]}-2026')
    bottom_legend(fig, ncol=2, fontsize=9)
    save_fig('ch7_sem_capital')
    return out


def nw_lrv(x):
    """Long-run variance (Newey-West, Bartlett kernel, lag floor(4 (n/100)^(2/9)))."""
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    m = int(np.floor(4 * (n / 100) ** (2 / 9)))
    s = x @ x / n
    for j in range(1, m + 1):
        s += 2 * (1 - j / (m + 1)) * (x[j:] @ x[:-j]) / n
    return s, m


def var_es_se(L, a_var=0.01, a_es=0.025):
    """Historical VaR: avar = a(1-a)/f(q)^2 (Gaussian kernel density, Silverman's rule);
    historical ES (Chen, 2008): influence function VaR + (L - VaR)_+/a - ES, avar = Var((L - VaR)_+)/a^2.
    HAC version: long-run variance of the indicator and of (L - VaR)_+, respectively."""
    L = np.asarray(L, float)
    n = len(L)
    v = np.quantile(L, 1 - a_var)
    f = stats.gaussian_kde(L, bw_method='silverman')(v)[0]
    ind = (L > v).astype(float)
    se_v = np.sqrt(a_var * (1 - a_var) / n) / f
    se_v_hac = np.sqrt(nw_lrv(ind)[0] / n) / f
    v2, e2 = hs_var_es(L, a_es)
    ex = np.maximum(L - v2, 0)
    se_e = ex.std(ddof=1) / (a_es * np.sqrt(n))
    lrv, m = nw_lrv(ex)
    se_e_hac = np.sqrt(lrv / n) / a_es
    return dict(n=n, var1=v, f=f, se_var=se_v, se_var_hac=se_v_hac, es=e2, var_es=v2, se_es=se_e,
                se_es_hac=se_e_hac, lag=m, ntail=a_es * n)


def ru_objective(L, v, alpha):
    return v + np.maximum(np.asarray(L) - v, 0).mean() / alpha


def min_es_portfolio(R, alpha=0.025):
    """Minimum ES on scenarios (Rockafellar & Uryasev, 2000): min v + 1/(alpha N) sum u_s,
    u_s >= -R_s w - v, u_s >= 0, sum w = 1, w >= 0 (no short sales)."""
    N, d = R.shape
    c = np.concatenate([np.zeros(d), [1.0], np.full(N, 1 / (alpha * N))])
    from scipy.sparse import csr_matrix, hstack as sh, identity
    A = sh([csr_matrix(-R), csr_matrix(-np.ones((N, 1))), -identity(N)]).tocsr()
    b = np.zeros(N)
    Aeq = np.concatenate([np.ones(d), [0.0], np.zeros(N)])[None, :]
    bounds = [(0, None)] * d + [(None, None)] + [(0, None)] * N
    out = optimize.linprog(c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=[1.0], bounds=bounds, method='highs')
    return out.x[:d], out.x[d], out.fun


def rearrangement(X, tol=1e-10, max_iter=1000):
    X = X.copy()
    for _ in range(max_iter):
        old = X.copy()
        for j in range(X.shape[1]):
            rest = X.sum(axis=1) - X[:, j]
            X[np.argsort(rest), j] = np.sort(X[:, j])[::-1]    # order opposite to the sum of the other columns
        if np.max(np.abs(X - old)) < tol:
            break
    return X


def worst_var_ra(Lcols, alpha=0.01, N=2000):
    """Worst-case VaR 1% of a sum with given marginals (Embrechts, Puccetti & Rueschendorf, 2013):
    discretise the tail of probability alpha of each marginal in N points (lower / upper) and rearrange;
    worst VaR ~ minimum of the row sums."""
    lo = np.column_stack([np.quantile(x, 1 - alpha + alpha * np.arange(N) / N) for x in Lcols])
    hi = np.column_stack([np.quantile(x, 1 - alpha + alpha * np.arange(1, N + 1) / N) for x in Lcols])
    return rearrangement(lo).sum(axis=1).min(), rearrangement(hi).sum(axis=1).min()


def fhs_bootstrap(r, alpha=0.01, B=999, seed=SEED, level=0.90, return_draws=False):
    """Christoffersen & Goncalves (2005): AR(1)-GARCH(1,1) paths with resampled residuals;
    re-estimate the parameters on each path; sigma_{T+1} from the ORIGINAL data with the bootstrap parameters;
    quantile from the standardised residuals of the bootstrap path."""
    rng = np.random.default_rng(seed)
    params, mu, sig = garch_filter(r)
    z = ((r - mu) / sig).dropna().values
    m1, s1 = garch_next(r, params)
    var0 = -m1 + s1 * np.quantile(-z, 1 - alpha)
    c, phi, om, al, be = params.values[:5]
    n = len(r)
    out = np.empty(B)
    parts = np.empty((B, 2))
    for b in range(B):
        zz = rng.choice(z, n)
        rs = np.empty(n)
        s2 = om / (1 - al - be)
        rp, ep = r.iloc[0], 0.0
        for t in range(n):
            s2 = om + al * ep ** 2 + be * s2 if t > 0 else s2
            e = np.sqrt(s2) * zz[t]
            rs[t] = c + phi * rp + e
            rp, ep = rs[t], e
        rs = pd.Series(rs, index=r.index)
        pb, mub, sgb = garch_filter(rs)
        zb = ((rs - mub) / sgb).dropna().values
        mb, sb = garch_next(r, pb)                    # ORIGINAL history, bootstrap parameters
        qb = np.quantile(-zb, 1 - alpha)
        out[b] = -mb + sb * qb
        parts[b] = [sb, qb]
    lo, hi = np.quantile(out, [(1 - level) / 2, (1 + level) / 2])
    # decomposition: quantile uncertainty only (sigma fixed) vs sigma uncertainty only (quantile fixed)
    q0 = np.quantile(-z, 1 - alpha)
    only_q = np.quantile(-m1 + s1 * parts[:, 1], [(1 - level) / 2, (1 + level) / 2])
    only_s = np.quantile(-m1 + parts[:, 0] * q0, [(1 - level) / 2, (1 + level) / 2])
    res = dict(var=var0, lo=lo, hi=hi, sd=out.std(ddof=1), B=B, level=level, sigma=s1,
                sig_lo=np.quantile(parts[:, 0], (1 - level) / 2), sig_hi=np.quantile(parts[:, 0], (1 + level) / 2),
                q=q0, q_lo=np.quantile(parts[:, 1], (1 - level) / 2), q_hi=np.quantile(parts[:, 1], (1 + level) / 2),
                onlyq_lo=only_q[0], onlyq_hi=only_q[1], onlys_lo=only_s[0], onlys_hi=only_s[1])
    return (res, out) if return_draws else res


def caviar_path(beta, r, v0):
    """VaR_t = b0 + b1 VaR_{t-1} + b2 |r_{t-1}| (positive VaR; quantile q_t = -VaR_t)."""
    b0, b1, b2 = beta
    x = b0 + b2 * np.abs(np.asarray(r, float)[:-1])
    rest = signal.lfilter([1.0], [1.0, -b1], x, zi=[b1 * v0])[0]
    return np.concatenate([[v0], rest])


def pinball(r, q, alpha):
    """Mean pinball (tick) loss for the return quantile q: (alpha - 1{r < q})(r - q)."""
    return np.mean((alpha - (r < q)) * (r - q))


def caviar_fit(r, alpha=0.01, n_rand=10_000, n_best=10, seed=SEED):
    """Engle & Manganelli (2004): 10^4 starting vectors U(0,1), the best 10 refined by alternating simplex
    and quasi-Newton steps until convergence; starting VaR = empirical quantile of the first 300 observations."""
    r = np.asarray(r, float)
    v0 = -np.quantile(r[:300], alpha)
    obj = lambda b: pinball(r, -caviar_path(b, r, v0), alpha) if abs(b[1]) < 1 else 1e6  # noqa: E731
    rng = np.random.default_rng(seed)
    cand = rng.uniform(0, 1, (n_rand, 3))
    vals = np.array([obj(b) for b in cand])
    best = []
    for b in cand[np.argsort(vals)[:n_best]]:
        f_old = np.inf
        for _ in range(20):
            b = optimize.minimize(obj, b, method='Nelder-Mead', options=dict(xatol=1e-8, fatol=1e-12, maxiter=4000)).x
            b = optimize.minimize(obj, b, method='BFGS').x
            f = obj(b)
            if f_old - f < 1e-12:
                break
            f_old = f
        best.append((obj(b), b))
    f, b = min(best, key=lambda x: x[0])
    return b, v0, f


def caviar_study(name, est_end='2015-12-31', oos_start='2016-01-01', alpha=0.01, fhs=None):
    r = 100 * log_returns(name)
    b, v0, f = caviar_fit(r.loc[:est_end].values, alpha)
    v = pd.Series(caviar_path(b, r.values, v0), index=r.index)
    L = -r
    oos = r.loc[oos_start:].index
    out = dict(b0=b[0], b1=b[1], b2=b[2], n_est=int(len(r.loc[:est_end])), n_oos=int(len(oos)),
               hit_caviar=float((L.loc[oos] > v.loc[oos]).mean()),
               n_hit_caviar=int((L.loc[oos] > v.loc[oos]).sum()),
               pin_caviar=pinball(r.loc[oos].values, -v.loc[oos].values, alpha))
    if fhs is not None:
        d = fhs.loc[oos_start:]
        out.update(hit_fhs=float((d['loss'] > d['fhs_var']).mean()), n_hit_fhs=int((d['loss'] > d['fhs_var']).sum()),
                   pin_fhs=pinball(-d['loss'].values, -d['fhs_var'].values, alpha),
                   hit_hs=float((d['loss'] > d['hs_var']).mean()), n_hit_hs=int((d['loss'] > d['hs_var']).sum()),
                   pin_hs=pinball(-d['loss'].values, -d['hs_var'].values, alpha))
        cov = d.loc['2020-02-20':'2020-04-30']
        vc = v.loc[cov.index]
        out.update(covid_hit_caviar=int((cov['loss'] > vc).sum()), covid_hit_fhs=int((cov['loss'] > cov['fhs_var']).sum()),
                   covid_hit_hs=int((cov['loss'] > cov['hs_var']).sum()),
                   covid_max_caviar=float(vc.max()), covid_max_fhs=float(cov['fhs_var'].max()),
                   covid_max_hs=float(cov['hs_var'].max()), covid_days=int(len(cov)))
    return out, v


def extremal_index(L, tail=0.05):
    """Intervals estimator: inter-exceedance times T_i; declustering with the C - 1 largest times,
    C = floor(theta N) + 1 (at most N); with ties, C is lowered until T_(C-1) > T_(C) (Ferro & Segers, 2003,
    Section 4); GPD on the cluster maxima."""
    L = np.asarray(L, float)
    u = np.quantile(L, 1 - tail)
    S = np.flatnonzero(L > u)
    T = np.diff(S).astype(float)
    N = len(S)
    if T.max() <= 2:
        th = 2 * T.sum() ** 2 / ((N - 1) * (T ** 2).sum())
    else:
        th = 2 * (T - 1).sum() ** 2 / ((N - 1) * ((T - 1) * (T - 2)).sum())
    th = min(1.0, th)
    C = min(int(np.floor(th * N)) + 1, N)                        # at most one cluster per exceedance
    cut = np.sort(T)[::-1][C - 2] if C > 1 else np.inf           # the (C-1)-th largest time
    breaks = np.flatnonzero(T > cut) if np.sum(T >= cut) > C - 1 else np.flatnonzero(T >= cut)
    groups = np.split(np.arange(N), breaks + 1)
    cmax = np.array([L[S[g]].max() for g in groups])
    xi, _, beta = stats.genpareto.fit(cmax - u, floc=0)
    fc = dict(u=u, xi=xi, beta=beta, nu=len(cmax), n=len(L))
    f = gpd_fit(L, tail)
    return dict(u=u, N=N, theta=th, n_clusters=len(groups), mean_size=N / len(groups),
                xi=f['xi'], se_xi=gpd_se(f)[0], xi_c=xi, se_xi_c=gpd_se(fc)[0], beta_c=beta,
                se_xi_eff=(1 + f['xi']) / np.sqrt(th * N))


def seminar_extras():
    out = {}
    # Seminar A8: asymptotic variance of the historical ES 2.5% under N(0, 1): Var((L - v)_+) / alpha^2, truncated Normal moments
    a = 0.025
    v = stats.norm.ppf(1 - a)
    m1 = stats.norm.pdf(v) - v * a                       # E[(L - v)_+]
    m2 = (1 + v ** 2) * a - v * stats.norm.pdf(v)        # E[(L - v)_+^2]
    avar = (m2 - m1 ** 2) / a ** 2
    out['a8'] = dict(v=v, phi=stats.norm.pdf(v), m1=m1, m2=m2, var=m2 - m1 ** 2, avar=avar,
                     se500=np.sqrt(avar / 500), se500_pct=1.2 * np.sqrt(avar / 500), es=stats.norm.pdf(v) / a)
    # Seminar A6: monotonicity of the Cornish-Fisher map for z in [z_0.1%, 0]: derivative a z^2 + b z + c > 0
    def dmin(S, K):
        z = np.linspace(stats.norm.ppf(0.001), 0, 20001)
        return (1 + z * S / 3 + (3 * z ** 2 - 3) * K / 24 - (6 * z ** 2 - 5) * S ** 2 / 36).min()
    S = -0.5
    kmax = optimize.brentq(lambda K: dmin(S, K), 0.1, 30)
    kmax0 = optimize.brentq(lambda K: dmin(0.0, K), 0.1, 30)
    out['a6'] = dict(kmax=kmax, kmax0=kmax0, d3=dmin(S, 3.0), d10=dmin(S, 10.0))
    # Seminar A1: asymptotic SE of the historical VaR 1% under N(0.04, 1.2^2)
    z = stats.norm.ppf(0.01)
    out['a1'] = dict(f=stats.norm.pdf(z) / 1.2, se500=1.2 * np.sqrt(0.01 * 0.99 / 500) / stats.norm.pdf(z),
                     se6670=1.2 * np.sqrt(0.01 * 0.99 / 6670) / stats.norm.pdf(z))
    # Seminar B1 Extended: BET, VaR 1%: asymptotic interval (kernel density) and block bootstrap (20 days)
    L = (-100 * log_returns('bet')).values
    for tag, x in [('full', L), ('w500', L[-500:])]:
        se = var_es_se(x)
        ci_blk, _ = boot_ci(x, lambda y: hs_var_es(y, 0.01)[0], block=20)
        out[f'b1_{tag}'] = dict(var1=se['var1'], f=se['f'], se=se['se_var'], se_hac=se['se_var_hac'],
                                lo=se['var1'] - 1.96 * se['se_var'], hi=se['var1'] + 1.96 * se['se_var'],
                                lo_hac=se['var1'] - 1.96 * se['se_var_hac'], hi_hac=se['var1'] + 1.96 * se['se_var_hac'],
                                blk_lo=ci_blk[0], blk_hi=ci_blk[1])
    return out


def _style():
    plt.rcParams['font.size'] = 10.5
    plt.rcParams['axes.titlesize'] = 10.5
    plt.rcParams['axes.labelsize'] = 10.5
    plt.rcParams['xtick.labelsize'] = 9.5
    plt.rcParams['ytick.labelsize'] = 9.5
    plt.rcParams['legend.fontsize'] = 9.5


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=9):
    """Legend below the figure (outside the axes), after tight_layout."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, lab in zip(*ax.get_legend_handles_labels()):
                if lab not in seen and not lab.startswith('_'):
                    handles.append(h)
                    labels.append(lab)
                    seen.add(lab)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False,
               fontsize=fontsize)


def fig_bet_data():
    """B1, before estimation: BET price and daily losses, with the window of the last 500 days."""
    _style()
    P = load_close('bet')
    L = -100 * np.log(P).diff().dropna()
    w0 = L.index[-500]
    fig, axes = plt.subplots(2, 1, figsize=(W_FIG, 3.0), sharex=True, gridspec_kw={'height_ratios': [1, 1.1]})
    ax = axes[0]
    ax.plot(P.index, P.values, color=MainBlue, lw=0.8, label='BET close (log scale)')
    ax.set_yscale('log')
    ax.set_ylabel('Index points')
    ax.axvspan(w0, P.index[-1], color=Amber, alpha=0.18, lw=0, label='Last 500 returns')
    ax = axes[1]
    ax.vlines(L.index, 0, L.clip(lower=0).values, color=Teal, lw=0.5, label='Daily log loss (gains set to 0)')
    ax.axvspan(w0, L.index[-1], color=Amber, alpha=0.18, lw=0)
    top = L.nlargest(60)
    d08 = top.loc['2008'].idxmax()
    d20 = top.loc['2020'].idxmax()
    for d, lab in [(d08, 'Oct 2008'), (d20, 'Mar 2020')]:
        ax.annotate(f'{lab}: {L.loc[d]:.1f}%', xy=(d, L.loc[d]), xytext=(8, -2), textcoords='offset points',
                    fontsize=9, color=IDAred)
    ax.set_ylabel('Loss (%)')
    bottom_legend(fig, ncol=3)
    save_fig('ch7_sem_bet_data')
    return dict(first=str(P.index[0].date()), last=str(P.index[-1].date()), p_first=float(P.iloc[0]),
                p_last=float(P.iloc[-1]), n_ret=int(len(L)), w500_start=str(w0.date()),
                max08=float(L.loc[d08]), d08=str(d08.date()), max20=float(L.loc[d20]), d20=str(d20.date()),
                maxall=float(L.max()), dmax=str(L.idxmax().date()), head=[(str(d.date()), float(v)) for d, v in P.iloc[:3].items()],
                ret_head=[(str(d.date()), float(v)) for d, v in (-L).iloc[:2].items()])


def fig_a1a2_tails(A1, A2):
    """A1-A2: the loss under the Normal distribution and under a t4 with the same standard deviation; VaR and ES at 1% and 2.5%."""
    _style()
    mu, sig, s, nu = 0.04, 1.2, A2['s'], 4
    x = np.linspace(0.0, 8.0, 800)
    sn = stats.norm.sf(x, -mu, sig)                     # L = -X ~ N(-mu, sigma^2): P(L > x)
    st = stats.t.sf((x + mu) / s, nu)
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.3))
    ax = axes[0]
    ax.semilogy(x, sn, color=MainBlue, label='Normal distribution, sd 1.2%')
    ax.semilogy(x, st, color=IDAred, label='Student-t, 4 df, sd 1.2%')
    for a_, ls in [(0.025, '--'), (0.01, ':')]:
        ax.axhline(a_, color=Gray, lw=0.6, ls=ls)
    ax.plot([A1['var1']], [0.01], 'o', color=MainBlue, ms=4)
    ax.plot([A2['var1']], [0.01], 'o', color=IDAred, ms=4)
    ax.plot([A1['es1']], [0.01], 'x', color=MainBlue, ms=5, mew=1.2)
    ax.plot([A2['es1']], [0.01], 'x', color=IDAred, ms=5, mew=1.2)
    ax.plot([], [], 'o', color='black', ms=4, label='VaR 1%')
    ax.plot([], [], 'x', color='black', ms=5, mew=1.2, label='ES 1%')
    ax.text(0.2, 0.013, '1%', fontsize=9, color='black')
    ax.text(0.2, 0.031, '2.5%', fontsize=9, color='black')
    ax.set_xlim(0, 8)
    ax.set_ylim(1e-4, 0.6)
    ax.set_xlabel('Daily loss x (%)')
    ax.set_ylabel('P(L > x)')
    ax.set_title('Tail probability of the loss')
    ax = axes[1]
    labs = ['VaR 2.5%', 'ES 2.5%', 'VaR 1%', 'ES 1%']
    vn = [A1['var2_5'], A1['es2_5'], A1['var1'], A1['es1']]
    vt = [A2['var2_5'], A2['es2_5'], A2['var1'], A2['es1']]
    xx = np.arange(4)
    ax.bar(xx - 0.19, vn, 0.36, color=MainBlue)
    ax.bar(xx + 0.19, vt, 0.36, color=IDAred)
    for i in range(4):
        ax.text(xx[i] - 0.19, vn[i] + 0.05, f'{vn[i]:.2f}', ha='right', fontsize=8, color=MainBlue)
        ax.text(xx[i] + 0.19, vt[i] + 0.05, f'{vt[i]:.2f}', ha='left', fontsize=8, color=IDAred)
    ax.set_xticks(xx)
    ax.set_xticklabels(labs, fontsize=8.5)
    ax.set_ylabel('% of the position')
    ax.set_ylim(0, 4.6)
    ax.set_title('Risk measures')
    bottom_legend(fig, ncol=4)
    save_fig('ch7_sem_a1a2_tails')


def fig_a3_mass():
    """A3: which probability mass enters ES 5% and ES 2.5% (loss 60 with p = 3%, otherwise 0)."""
    _style()
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.2), gridspec_kw={'width_ratios': [1, 1.35]})
    ax = axes[0]
    ax.bar([0, 60], [97, 3], width=9, color=[MainBlue, IDAred])
    ax.text(0, 99, '97%', ha='center', fontsize=9, color=MainBlue)
    ax.text(60, 5, '3%', ha='center', fontsize=9, color=IDAred)
    ax.set_xticks([0, 60])
    ax.set_xlabel('Loss (monetary units)')
    ax.set_ylabel('Probability (%)')
    ax.set_ylim(0, 110)
    ax.set_title('Loss distribution')
    ax = axes[1]
    rows = [('ES 5%', 3.0, 2.0, 36), ('ES 2.5%', 2.5, 0.0, 60)]
    for i, (lab, m60, m0, es) in enumerate(rows):
        ax.barh(i, m60, color=IDAred, height=0.5, label='mass at loss 60' if i == 0 else None)
        ax.barh(i, m0, left=m60, color=MainBlue, height=0.5, label='mass at loss 0' if i == 0 else None)
        ax.text(5.15, i, f'ES = {es}', va='center', fontsize=9, color='black')
    ax.set_yticks([0, 1])
    ax.set_yticklabels([r[0] for r in rows])
    ax.set_xlim(0, 6.6)
    ax.set_xlabel('Tail probability used (%)')
    ax.set_title('Probability mass in the worst α of outcomes')
    bottom_legend(fig, ncol=2)
    save_fig('ch7_sem_a3_mass')


def fig_a4_var_alpha(p=0.007):
    """A4: VaR and ES as functions of alpha for one bond (x2) and for the portfolio A + B."""
    _style()
    al = np.linspace(0.002, 0.03, 700)
    one = [discrete_var_es([0, 100], [1 - p, p], a) for a in al]
    por = [discrete_var_es([0, 100, 200], [(1 - p) ** 2, 2 * p * (1 - p), p ** 2], a) for a in al]
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.3), sharex=True)
    lo, hi = p, 1 - (1 - p) ** 2
    for ax, j, ttl in [(axes[0], 0, 'VaR'), (axes[1], 1, 'ES')]:
        ax.axvspan(100 * lo, 100 * hi, color=Amber, alpha=0.2, lw=0, label='VaR not subadditive')
        ax.plot(100 * al, [2 * v[j] for v in one], color=MainBlue, label='VaR(A) + VaR(B), ES(A) + ES(B)')
        ax.plot(100 * al, [v[j] for v in por], color=IDAred, ls='--', label='Portfolio A + B')
        ax.set_title(ttl)
        ax.set_xlabel('Tail probability alpha (%)')
    axes[0].set_ylabel('Loss (monetary units)')
    axes[0].axvline(1.0, color=Gray, lw=0.6, ls=':')
    axes[1].axvline(1.0, color=Gray, lw=0.6, ls=':')
    bottom_legend(fig, ncol=3)
    save_fig('ch7_sem_a4_var_alpha')
    return dict(p_one=2 * p * (1 - p), lo=lo, hi=hi)


def fig_a5_shares(A5):
    """A5: money weights vs VaR shares; summed stand-alone VaR vs diversified VaR."""
    _style()
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.2), gridspec_kw={'width_ratios': [1.3, 1]})
    ax = axes[0]
    xx = np.arange(2)
    wgt = 100 * np.array(A5['w']) / np.sum(A5['w'])
    sh = 100 * np.array(A5['share'])
    ax.bar(xx - 0.19, wgt, 0.36, color=Teal, label='Money weight')
    ax.bar(xx + 0.19, sh, 0.36, color=MainBlue, label='Share of Normal VaR 1%')
    for i in range(2):
        ax.text(xx[i] - 0.19, wgt[i] + 2, f'{wgt[i]:.0f}%', ha='center', fontsize=9, color='black')
        ax.text(xx[i] + 0.19, sh[i] + 2, f'{sh[i]:.1f}%', ha='center', fontsize=9, color='black')
    ax.set_xticks(xx)
    ax.set_xticklabels(['Equity', 'Bonds'])
    ax.set_ylim(0, 100)
    ax.set_ylabel('%')
    ax.set_title('Money versus risk')
    ax = axes[1]
    vals = [A5['var_ind'][0], A5['var_ind'][1], A5['var_p_amt']]
    ax.bar([0], [vals[0]], 0.5, color=Orange, label='Stand-alone VaR, equity')
    ax.bar([0], [vals[1]], 0.5, bottom=[vals[0]], color=Amber, label='Stand-alone VaR, bonds')
    ax.bar([1], [vals[2]], 0.5, color=IDAred, label='Portfolio VaR 1%')
    ax.text(0, vals[0] + vals[1] + 400, f'{vals[0] + vals[1]:,.0f}', ha='center', fontsize=9)
    ax.text(1, vals[2] + 400, f'{vals[2]:,.0f}', ha='center', fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Sum', 'Portfolio'])
    ax.set_ylabel('EUR')
    ax.set_ylim(0, 29000)
    ax.set_title('Diversification')
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_a5_shares')


def risk_parity(S, tol=1e-12):
    """Non-negative, fully invested weights with equal contributions to the Normal VaR: w_i (S w)_i = constant."""
    from scipy import optimize
    d = S.shape[0]

    def obj(y):
        w = np.exp(y) / np.exp(y).sum()
        rc = w * (S @ w)
        return ((rc / rc.sum() - 1 / d) ** 2).sum()
    res = optimize.minimize(obj, np.zeros(d), method='Nelder-Mead', options=dict(xatol=1e-10, fatol=tol, maxiter=20000))
    w = np.exp(res.x) / np.exp(res.x).sum()
    return w


def fig_b5_shares(B5):
    """B5: equal weights vs shares of the Normal VaR 1% and of the historical ES 2.5%; plus the risk-parity weights."""
    _style()
    names = ['SPY', 'TLT', 'GLD', 'Bitcoin']
    lr = joint_returns(['SPY', 'TLT', 'GLD', 'BTC'], start='2014-09-18')
    R = 100 * (np.exp(lr) - 1)
    S = np.cov(R.values.T)
    w_rp = risk_parity(S)
    z = stats.norm.ppf(0.99)
    sp_rp = np.sqrt(w_rp @ S @ w_rp)
    Lp = -(R.values @ w_rp)
    v_rp, e_rp = hs_var_es(Lp, 0.025)
    ces_rp = -(R.values[Lp >= v_rp] * w_rp).mean(axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.3), gridspec_kw={'width_ratios': [1.6, 1]})
    ax = axes[0]
    xx = np.arange(4)
    for k, (vals, c, lab) in enumerate([(25 * np.ones(4), Teal, 'Money weight'),
                                        (100 * np.array(B5['share']), MainBlue, 'Share of Normal VaR 1%'),
                                        (100 * np.array(B5['ces_share']), IDAred, 'Share of historical ES 2.5%')]):
        ax.bar(xx + (k - 1) * 0.27, vals, 0.25, color=c, label=lab)
    for i in range(4):
        ax.text(xx[i] + 0.27, 100 * B5['ces_share'][i] + 2, f"{100 * B5['ces_share'][i]:.0f}", ha='center', fontsize=9,
                color=IDAred)
    ax.set_xticks(xx)
    ax.set_xticklabels(names)
    ax.set_ylabel('%')
    ax.set_ylim(0, 95)
    ax.set_title('Equal weights, 2014-2026')
    ax = axes[1]
    ax.bar([0], [B5['undiv']], 0.5, color=Orange, label='Sum of stand-alone VaR 1%')
    ax.bar([1], [B5['var_p']], 0.5, color=MainBlue, label='Portfolio VaR 1% (Normal)')
    ax.bar([2], [B5['es_p']], 0.5, color=IDAred, label='Portfolio ES 2.5% (historical)')
    for i, v in enumerate([B5['undiv'], B5['var_p'], B5['es_p']]):
        ax.text(i, v + 0.08, f'{v:.2f}', ha='center', fontsize=9)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Sum', 'VaR', 'ES'])
    ax.set_ylabel('% of portfolio value')
    ax.set_ylim(0, 5)
    ax.set_title('Portfolio totals')
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_b5_shares')
    rc = w_rp * (S @ w_rp)
    return dict(w_rp=w_rp, var_rp=z * sp_rp, share_rp=rc / rc.sum(), es_rp=e_rp, ces_rp_share=ces_rp / e_rp,
                sd=np.sqrt(np.diag(S)), inv_vol=(1 / np.sqrt(np.diag(S))) / (1 / np.sqrt(np.diag(S))).sum(),
                Sw_spy=(S @ np.full(4, 0.25))[0], var_p=B5['var_p'])


def fig_b5_rp(B5, RP):
    """B5 Extended: risk-parity weights (Normal VaR) vs 1/sigma and the resulting risk shares."""
    _style()
    names = ['SPY', 'TLT', 'GLD', 'Bitcoin']
    xx = np.arange(4)
    fig, ax = plt.subplots(figsize=(W_FIG * 0.85, 2.3))
    for k, (vals, c, lab) in enumerate([(100 * np.array(RP['w_rp']), MainBlue, 'Risk-parity weight'),
                                        (100 * np.array(RP['inv_vol']), Teal, 'Inverse-volatility weight'),
                                        (100 * np.array(RP['ces_rp_share']), IDAred, 'Share of historical ES 2.5% at risk parity')]):
        ax.bar(xx + (k - 1) * 0.27, vals, 0.25, color=c, label=lab)
        for i in range(4):
            ax.text(xx[i] + (k - 1) * 0.27, vals[i] + 0.8, f'{vals[i]:.0f}', ha='center', fontsize=9, color='black')
    ax.axhline(25, color=Gray, lw=0.6, ls=':')
    ax.set_xticks(xx)
    ax.set_xticklabels(names)
    ax.set_ylabel('%')
    ax.set_ylim(0, 45)
    bottom_legend(fig, ncol=2, fontsize=9.5)
    save_fig('ch7_sem_b5_rp')


def fig_a6_cf(S=-0.5):
    """A6: the Cornish-Fisher map z -> z~(z) and its derivative for K = 3 and K = 10."""
    _style()
    z = np.linspace(stats.norm.ppf(0.001), 1.0, 500)
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.3))
    for K, c in [(3.0, MainBlue), (10.0, IDAred)]:
        axes[0].plot(z, cf_quantile(z, S, K), color=c, label=f'K = {K:.0f}')
        d = 1 + z * S / 3 + (z ** 2 - 1) * K / 8 - (6 * z ** 2 - 5) * S ** 2 / 36
        axes[1].plot(z, d, color=c)
    axes[0].plot(z, z, color=Gray, lw=0.6, ls=':')
    axes[0].set_xlabel('Normal quantile z')
    axes[0].set_ylabel(r'Cornish-Fisher quantile $\tilde z$')
    axes[0].set_title(r'The map $z \mapsto \tilde z(z)$, $S = -0.5$')
    axes[1].axhline(0, color=Gray, lw=0.6)
    axes[1].axvspan(stats.norm.ppf(0.001), 0, color=Amber, alpha=0.15, lw=0, label=r'Interval $[z_{0.1\%}, 0]$')
    axes[1].set_xlabel('Normal quantile z')
    axes[1].set_ylabel(r'Derivative $d\tilde z/dz$')
    axes[1].set_title('Negative slope = not a quantile')
    bottom_legend(fig, ncol=3)
    save_fig('ch7_sem_a6_cf')


def fig_a7_ru(p=0.007, alpha=0.01):
    """A7: the Rockafellar-Uryasev function F(v) = v + E[(L - v)+]/alpha for the A4 portfolio and for one bond."""
    _style()
    v = np.linspace(-10, 220, 2301)
    vals_p, pr_p = np.array([0, 100, 200.]), np.array([(1 - p) ** 2, 2 * p * (1 - p), p ** 2])
    vals_a, pr_a = np.array([0, 100.]), np.array([1 - p, p])
    F = lambda vals, pr: v + (np.maximum(vals[None, :] - v[:, None], 0) * pr).sum(axis=1) / alpha   # noqa: E731
    Fp, Fa = F(vals_p, pr_p), F(vals_a, pr_a)
    fig, ax = plt.subplots(figsize=(W_FIG * 0.8, 2.3))
    ax.plot(v, Fp, color=IDAred, label=r'$F(v)$, portfolio A + B')
    ax.plot(v, 2 * Fa, color=MainBlue, ls='--', label=r'$F_A(v) + F_B(v)$, stand-alone bonds')
    vp = 100.0
    fp = vp + ((np.maximum(vals_p - vp, 0) * pr_p).sum()) / alpha
    ax.plot([vp], [fp], 'o', color=IDAred, ms=4)
    ax.annotate(f'minimum {fp:.2f} at v = VaR = 100', xy=(vp, fp), xytext=(8, -16), textcoords='offset points',
                fontsize=9, color=IDAred)
    ax.text(40, 92, 'slope -0.3951', fontsize=9, color=IDAred, ha='center')
    ax.text(185, 150, 'slope 0.9951', fontsize=9, color=IDAred, ha='left')
    ax.set_xlabel(r'$v$')
    ax.set_ylabel(r'$F(v)$')
    ax.set_ylim(70, 250)
    ax.set_xlim(-10, 220)
    bottom_legend(fig, ncol=2)
    save_fig('ch7_sem_a7_ru')
    return dict(fmin=fp, s1=1 - (1 - (1 - p) ** 2) / alpha, s2=1 - p ** 2 / alpha, fa_min=2 * (0 + 100 * p / alpha))


def fig_a8_sim(R=4000, alpha=0.025, seed=SEED):
    """A8 / B6: sampling distribution of the historical ES 2.5%, Monte Carlo simulation (i.i.d.) vs asymptotic SE."""
    _style()
    rng = np.random.default_rng(seed)
    sig, nu = 1.2, 4
    s_t = sig * np.sqrt((nu - 2) / nu)
    v_n = sig * stats.norm.ppf(1 - alpha)
    es_n = sig * stats.norm.pdf(stats.norm.ppf(1 - alpha)) / alpha
    q_t = stats.t.ppf(1 - alpha, nu)
    v_t = s_t * q_t
    es_t = s_t * stats.t.pdf(q_t, nu) / alpha * (nu + q_t ** 2) / (nu - 1)
    # asymptotic variance Var((L - v)+)/alpha^2, by numerical integration
    from scipy import integrate as _int
    def avar(pdf, v, lim):
        m1 = _int.quad(lambda x: (x - v) * pdf(x), v, lim, limit=400)[0]
        m2 = _int.quad(lambda x: (x - v) ** 2 * pdf(x), v, lim, limit=400)[0]
        return (m2 - m1 ** 2) / alpha ** 2
    av_n = avar(lambda x: stats.norm.pdf(x, 0, sig), v_n, np.inf)
    av_t = avar(lambda x: stats.t.pdf(x / s_t, nu) / s_t, v_t, np.inf)
    ns = [250, 500, 1000, 2500, 5000]
    sd_n, sd_t, draws = [], [], {}
    for n in ns:
        en = np.array([hs_var_es(rng.normal(0, sig, n), alpha)[1] for _ in range(R)])
        et = np.array([hs_var_es(s_t * rng.standard_t(nu, n), alpha)[1] for _ in range(R)])
        sd_n.append(en.std(ddof=1))
        sd_t.append(et.std(ddof=1))
        if n == 500:
            draws = dict(n=en, t=et)
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.3))
    ax = axes[0]
    bins = np.linspace(1.5, 5.0, 60)
    ax.hist(draws['n'], bins=bins, density=True, color=MainBlue, alpha=0.5, label='Normal distribution')
    ax.hist(draws['t'], bins=bins, density=True, color=IDAred, alpha=0.5, label='Student-t, 4 df')
    xx = np.linspace(1.5, 5.0, 300)
    ax.plot(xx, stats.norm.pdf(xx, es_n, np.sqrt(av_n / 500)), color=MainBlue, lw=1.2, label='Asymptotic Normal law')
    ax.axvline(es_n, color=MainBlue, lw=0.8, ls='--')
    ax.axvline(es_t, color=IDAred, lw=0.8, ls='--')
    ax.set_xlabel('Historical ES 2.5% (%), n = 500')
    ax.set_ylabel('Density')
    ax.set_title(f'{R} simulated samples')
    ax = axes[1]
    ax.loglog(ns, sd_n, 'o', color=MainBlue, ms=4)
    ax.loglog(ns, sd_t, 's', color=IDAred, ms=4)
    nn = np.array([200, 6000])
    ax.loglog(nn, np.sqrt(av_n / nn), color=MainBlue, lw=0.9)
    ax.loglog(nn, np.sqrt(av_t / nn), color=IDAred, lw=0.9, ls='--')
    ax.plot([], [], 'o', color='black', ms=4, label='Simulated SD')
    ax.plot([], [], color='black', lw=0.9, label='Asymptotic SE')
    ax.set_xlabel('Sample size n')
    ax.set_ylabel('SD of ES estimate (pp)')
    ax.set_title('Precision grows like sqrt(n)')
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_a8_sim')
    i5 = ns.index(500)
    return dict(R=R, es_n=es_n, es_t=es_t, se_n500=float(np.sqrt(av_n / 500)), se_t500=float(np.sqrt(av_t / 500)),
                sd_n500=float(sd_n[i5]), sd_t500=float(sd_t[i5]), sd_n5000=float(sd_n[-1]),
                se_n5000=float(np.sqrt(av_n / 5000)), mean_n500=float(draws['n'].mean()), mean_t500=float(draws['t'].mean()))


def _ci_panel(ax, rows, title, xlab, band=None):
    """Point-and-interval chart; rows = (label, estimate, lo, hi, colour, text on the right)."""
    for i, (lab, est, lo, hi, c, txt) in enumerate(rows):
        y = len(rows) - 1 - i
        if band is not None:
            ax.fill_betweenx([y - 0.3, y + 0.3], est - band, est + band, color=Amber, alpha=0.25, lw=0)
        ax.plot([lo, hi], [y, y], color=c, lw=2.2, solid_capstyle='butt')
        ax.plot([est], [y], 'o', color=c, ms=4.5)
        if txt:
            ax.text(hi + 0.04, y, txt, va='center', fontsize=9, color='black')
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows][::-1], fontsize=9)
    ax.set_xlabel(xlab)
    ax.set_title(title)


def fig_b1_ci(B1, SX):
    """B1: 95% intervals for the BET VaR 1% on two samples and four methods."""
    _style()
    fig, axes = plt.subplots(1, 2, figsize=(W_FIG, 2.35), sharex=True)
    for ax, k, t in [(axes[0], 'full', f"2000-2026 ({B1['full']['n_beyond']} beyond VaR)"),
                     (axes[1], 'w500', f"Last 500 days ({B1['w500']['n_beyond']} beyond)")]:
        b, s = B1[k], SX[f'b1_{k}']
        rows = [('i.i.d. bootstrap', b['var1'], b['ci_lo'], b['ci_hi'], MainBlue, f"{b['ci_hi'] - b['ci_lo']:.2f}"),
                ('Block bootstrap', b['var1'], s['blk_lo'], s['blk_hi'], IDAred, f"{s['blk_hi'] - s['blk_lo']:.2f}"),
                ('Asymptotic i.i.d.', b['var1'], s['lo'], s['hi'], Teal, f"{s['hi'] - s['lo']:.2f}"),
                ('Asymptotic HAC', b['var1'], s['lo_hac'], s['hi_hac'], Purple, f"{s['hi_hac'] - s['lo_hac']:.2f}")]
        _ci_panel(ax, rows, t, 'VaR 1% of BET (%)')
    axes[1].set_yticklabels([])
    axes[0].set_xlim(1.8, 5.3)
    fig.text(0.5, -0.02, 'Dot: historical VaR 1%; bar: 95% interval; number: interval width in percentage points',
             ha='center', fontsize=9, color='black')
    plt.tight_layout()
    save_fig('ch7_sem_b1_ci')


def es_replicate_example(L, alpha=0.025, block=20, seed=SEED):
    """A single bootstrap resample (i.i.d. and by blocks), with VaR and tail membership recomputed."""
    rng = np.random.default_rng(seed)
    L = np.asarray(L)
    n = len(L)
    out = {}
    x = L[rng.integers(0, n, n)]
    v, e = hs_var_es(x, alpha)
    out['iid'] = dict(var=v, es=e, k=int((x >= v).sum()))
    starts = rng.integers(0, n - block + 1, int(np.ceil(n / block)))
    x = np.concatenate([L[s:s + block] for s in starts])[:n]
    v, e = hs_var_es(x, alpha)
    out['blk'] = dict(var=v, es=e, k=int((x >= v).sum()), nblocks=int(len(starts)), first_starts=[int(s) for s in starts[:3]])
    v0, e0 = hs_var_es(L, alpha)
    out['orig'] = dict(var=v0, es=e0, k=int((L >= v0).sum()), nalpha=n * alpha)
    return out


def fig_b6_ci(B6):
    """B6: 95% intervals for ES 2.5% (i.i.d. vs blocks), with a band of +/-0.3 percentage points."""
    _style()
    fig, ax = plt.subplots(figsize=(W_FIG, 2.6))
    labs = {'sp500_full': 'S&P 500, 2000-2026', 'sp500_w500': 'S&P 500, last 500', 'bet_full': 'BET, 2000-2026',
            'bet_w500': 'BET, last 500'}
    rows = []
    for k in ['sp500_full', 'sp500_w500', 'bet_full', 'bet_w500']:
        b = B6[k]
        rows.append((labs[k] + ', i.i.d.', b['es'], b['iid_lo'], b['iid_hi'], MainBlue, f"{b['iid_hi'] - b['iid_lo']:.2f}"))
        rows.append((labs[k] + ', blocks', b['es'], b['blk_lo'], b['blk_hi'], IDAred, f"{b['blk_hi'] - b['blk_lo']:.2f}"))
    _ci_panel(ax, rows, '', 'ES 2.5% (%): dot = estimate, bar = 95% bootstrap interval', band=0.3)
    ax.fill_between([], [], color=Amber, alpha=0.25, label='Target precision: estimate +/- 0.3 pp')
    ax.plot([], [], color=MainBlue, lw=2.2, label='i.i.d. bootstrap')
    ax.plot([], [], color=IDAred, lw=2.2, label='Moving-block bootstrap, 20 days')
    ax.set_xlim(1.6, 5.8)
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_b6_ci')


def clusters_bet(L, tail=0.05):
    """Threshold exceedances, clusters (Ferro & Segers, 2003, tie rule) and cluster maxima."""
    L = np.asarray(L, float)
    u = np.quantile(L, 1 - tail)
    S = np.flatnonzero(L > u)
    T = np.diff(S).astype(float)
    N = len(S)
    th = min(1.0, 2 * (T - 1).sum() ** 2 / ((N - 1) * ((T - 1) * (T - 2)).sum()))
    C = min(int(np.floor(th * N)) + 1, N)
    cut = np.sort(T)[::-1][C - 2]
    breaks = np.flatnonzero(T > cut) if np.sum(T >= cut) > C - 1 else np.flatnonzero(T >= cut)
    groups = np.split(np.arange(N), breaks + 1)
    return u, S, groups, th, cut


def fig_b2_ext():
    """B2 Extended: timeline of exceedances and clusters, mean excess, stability of VaR 0.1% across thresholds."""
    _style()
    Ls = -100 * log_returns('bet')
    L = Ls.values
    u, S, groups, th, cut = clusters_bet(L)
    fig = plt.figure(figsize=(W_FIG, 3.4))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.05, 1])
    ax = fig.add_subplot(gs[0, :])
    dates = Ls.index[S]
    cols = [MainBlue, Teal]
    for j, g in enumerate(groups):
        ax.plot(dates[g], L[S[g]], 'o', ms=2.2, color=cols[j % 2], label='Exceedances (clusters alternate colour)' if j == 0 else None)
        m = g[np.argmax(L[S[g]])]
        ax.plot(dates[m], L[S[m]], 'o', ms=3.4, mfc='none', mec=IDAred, mew=0.8,
                label='Cluster maximum' if j == 0 else None)
    ax.axhline(u, color=Gray, lw=0.6, ls='--')
    ax.set_ylabel('Loss (%)')
    ax.set_title(f'BET losses above u = {u:.2f}%: {len(S)} exceedances, {len(groups)} clusters')
    ax2 = fig.add_subplot(gs[1, 0])
    us = np.quantile(L, np.linspace(0.80, 0.995, 40))
    me, cnt = mean_excess(L, us)
    ax2.plot(us, me, 'o', ms=2.5, color=Forest, label='Empirical mean excess')
    ax2.axvline(u, color=Gray, lw=0.6, ls='--')
    ax2.set_xlabel('Threshold u (%)')
    ax2.set_ylabel('Mean excess (%)')
    ax2.set_title('Mean excess plot')
    ax3 = fig.add_subplot(gs[1, 1])
    qs = np.linspace(0.10, 0.015, 18)
    v01 = [gpd_var_es(gpd_fit(L, q), 0.001)[0] for q in qs]
    ax3.plot(100 * qs, v01, 'o-', ms=2.5, color=IDAred, label='GPD VaR 0.1%')
    ax3.axhline(hs_var_es(L, 0.001)[0], color=Gray, lw=0.6, ls=':')
    ax3.set_xlabel('Share of losses above u (%)')
    ax3.set_ylabel('VaR 0.1% (%)')
    ax3.set_title('Extreme quantile vs threshold')
    ax3.set_ylim(8, 10.5)
    bottom_legend(fig, ncol=4, fontsize=9.5)
    save_fig('ch7_sem_b2_clusters')
    sizes = np.array([len(g) for g in groups])
    return dict(u=u, N=int(len(S)), n_clusters=int(len(groups)), theta=th, cut=float(cut), max_size=int(sizes.max()),
                n_single=int((sizes == 1).sum()), hs_var01=float(hs_var_es(L, 0.001)[0]))


def fig_btc_fhs3(B3):
    """B3: Bitcoin losses, 500-day HS VaR and FHS VaR with exceedances; conditional volatility; ACF of z^2."""
    _style()
    r = 100 * log_returns('btc')
    params, mu, sig = garch_filter(r)
    z = ((r - mu) / sig).dropna()
    roll = pd.read_csv(os.path.join(HERE, 'ch7_rolling_btc.csv'), index_col=0, parse_dates=True)
    rec = roll.loc['2024-09-18':]
    fig = plt.figure(figsize=(W_FIG, 3.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])
    ax = fig.add_subplot(gs[0, :])
    ax.bar(rec.index, rec['loss'].clip(lower=0), width=1, color=Teal, alpha=0.55, label='Daily loss (gains set to 0)')
    ax.plot(rec.index, rec['hs_var'], color=MainBlue, lw=1.0, label='HS VaR 1% (rolling 500 days)')
    ax.plot(rec.index, rec['fhs_var'], color=IDAred, lw=0.8, label='FHS VaR 1%')
    bh = rec[rec['loss'] > rec['hs_var']]
    bf = rec[rec['loss'] > rec['fhs_var']]
    ax.plot(bh.index, bh['loss'] + 0.6, 'v', ms=4, color=MainBlue, label=f'HS exceedance ({len(bh)})')
    ax.plot(bf.index, bf['loss'] + 1.6, 'v', ms=4, color=IDAred, label=f'FHS exceedance ({len(bf)})')
    ax.set_ylabel('Loss (%)')
    ax.set_title('Bitcoin, 18 Sep 2024 - 18 Sep 2026: one-day-ahead VaR 1%')
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(sig.loc['2017':].index, sig.loc['2017':].values, color=Purple, lw=0.7, label=r'GARCH volatility $\sigma_t$')
    ax2.set_ylabel(r'$\sigma_t$ (%)')
    import matplotlib.dates as mdates
    ax2.xaxis.set_major_locator(mdates.YearLocator(3))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax2.set_title('Conditional volatility')
    ax3 = fig.add_subplot(gs[1, 1])
    zz = z.values ** 2 - (z.values ** 2).mean()
    lags = np.arange(1, 21)
    acf2 = [np.corrcoef(zz[k:], zz[:-k])[0, 1] for k in lags]
    rr2 = (r.values - r.values.mean()) ** 2
    acf_raw = [np.corrcoef(rr2[k:], rr2[:-k])[0, 1] for k in lags]
    ax3.bar(lags - 0.2, acf_raw, 0.38, color=Orange, label='ACF of squared returns')
    ax3.bar(lags + 0.2, acf2, 0.38, color=Forest, label=r'ACF of squared residuals $\hat z_t^2$')
    band = 1.96 / np.sqrt(len(zz))
    ax3.axhspan(-band, band, color=LightGray, alpha=0.6, lw=0)
    ax3.set_xlabel('Lag (days)')
    ax3.set_title('Clustering removed?')
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_btc_fhs')
    return dict(n_hs=int(len(bh)), n_fhs=int(len(bf)), acf1_raw=float(acf_raw[0]), acf1_z=float(acf2[0]),
                max_acf_z=float(np.max(np.abs(acf2))), band=float(band), n_z=int(len(zz)))


def fig_b3_boot(draws, B3, boot):
    """B3 Extended: bootstrap distribution of tomorrow's FHS VaR 1% for Bitcoin."""
    _style()
    fig, ax = plt.subplots(figsize=(W_FIG * 0.85, 2.2))
    ax.hist(draws, bins=45, color=IDAred, alpha=0.55, label=f'{len(draws)} bootstrap forecasts')
    ax.axvspan(boot['lo'], boot['hi'], color=Amber, alpha=0.2, lw=0, label='90% percentile interval')
    ax.axvline(B3['fhs_var'], color=IDAred, lw=1.2, label=f"FHS VaR 1% = {B3['fhs_var']:.2f}%")
    ax.axvline(B3['hs500_var'], color=MainBlue, lw=1.2, ls='--', label=f"HS 500 days = {B3['hs500_var']:.2f}% (unconditional)")
    ax.set_xlabel('VaR 1% for 19 September 2026 (%)')
    ax.set_ylabel('Count')
    bottom_legend(fig, ncol=2, fontsize=9.5)
    save_fig('ch7_sem_b3_boot')


def fig_b4_methods(B4, B=1000):
    """B4: historical VaR with a bootstrap interval, Normal and Cornish-Fisher, by asset and level."""
    _style()
    keys = ['sp500', 'bet', 'btc', 'eurron', 'gold']
    labs = {'sp500': 'S&P 500', 'bet': 'BET', 'btc': 'Bitcoin', 'eurron': 'EUR/RON', 'gold': 'Gold'}
    ci25 = {}
    for k in keys:
        L = (-100 * log_returns(k)).values
        ci25[k] = boot_ci(L, lambda y: hs_var_es(y, 0.025)[0], B=B)[0]
    fig, axes = plt.subplots(1, 5, figsize=(W_FIG, 2.4))
    for ax, k in zip(axes, keys):
        b = B4[k]
        for j, (lvl, hs, lo, hi, n, cf) in enumerate([('1%', b['hs1'], b['ci_lo'], b['ci_hi'], b['n1'], b['cf1']),
                                                       ('2.5%', b['hs2_5'], ci25[k][0], ci25[k][1], b['n2_5'], b['cf2_5'])]):
            x = j
            ax.plot([x, x], [lo, hi], color=MainBlue, lw=2.4, solid_capstyle='butt')
            ax.plot([x], [hs], 'o', color=MainBlue, ms=4, label='Historical, 95% bootstrap CI' if (k == 'sp500' and j == 0) else None)
            ax.plot([x + 0.18], [n], 's', color=Orange, ms=3.5, label='Normal distribution' if (k == 'sp500' and j == 0) else None)
            ax.plot([x - 0.18], [cf], 'D', color=Purple, ms=3.5, label='Cornish-Fisher' if (k == 'sp500' and j == 0) else None)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['1%', '2.5%'], fontsize=9)
        ax.set_xlim(-0.5, 1.5)
        ax.set_title(labs[k], fontsize=9.5)
        ax.tick_params(axis='y', labelsize=7)
    axes[0].set_ylabel('Daily VaR (%)')
    axes[2].set_xlabel('Tail probability of VaR')
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_b4_methods')
    return {k: dict(lo=float(v[0]), hi=float(v[1])) for k, v in ci25.items()}


def rolling_conditional_bet():
    return rolling_conditional(100 * log_returns('bet'), 2016)


def fig_b7_caviar(CV):
    """B7: BET losses 2016-2026 with CAViaR, FHS and HS VaR; the COVID-19 window."""
    _style()
    try:
        from estimation_risk import caviar_path as _cp, pinball as _pb
    except ImportError:                              # in the Quantlet notebook these functions are already defined
        _cp, _pb = globals()['caviar_path'], globals()['pinball']
    r = 100 * log_returns('bet')
    v0 = -np.quantile(r.loc[:'2015-12-31'].values[:300], 0.01)
    v = pd.Series(_cp([CV['b0'], CV['b1'], CV['b2']], r.values, v0), index=r.index)
    fhs = rolling_conditional_bet()
    d = fhs.loc['2016-01-01':]
    L = d['loss']
    fig, axes = plt.subplots(2, 1, figsize=(W_FIG, 3.4), gridspec_kw={'height_ratios': [1.1, 1]})
    for ax, dd, ttl in [(axes[0], d, 'BET, 2016-2026: one-day-ahead VaR 1% out of sample'),
                        (axes[1], d.loc['2020-02-20':'2020-04-30'], 'COVID-19 window, 20 Feb - 30 Apr 2020')]:
        ax.bar(dd.index, dd['loss'].clip(lower=0), width=1 if ax is axes[1] else 2, color=Teal, alpha=0.55,
               label='Daily loss (gains set to 0)')
        ax.plot(dd.index, v.loc[dd.index], color=MainBlue, lw=0.9, label='CAViaR (SAV), parameters 2000-2015')
        ax.plot(dd.index, dd['fhs_var'], color=IDAred, lw=0.8, label='FHS, re-estimated every January')
        ax.plot(dd.index, dd['hs_var'], color=Orange, lw=0.9, label='HS, rolling 500 days')
        br = dd[dd['loss'] > v.loc[dd.index]]
        ax.plot(br.index, br['loss'] + 0.5, 'v', ms=3.5, color=MainBlue, label='CAViaR exceedance')
        ax.set_ylabel('Loss (%)')
        ax.set_title(ttl)
    bottom_legend(fig, ncol=3, fontsize=9.5)
    save_fig('ch7_sem_b7_caviar')
    return dict(n_hit_caviar=int((L > v.loc[d.index]).sum()), n_hit_fhs=int((L > d['fhs_var']).sum()),
                n_hit_hs=int((L > d['hs_var']).sum()), pin_caviar=float(_pb(-L.values, -v.loc[d.index].values, 0.01)),
                v0=float(v0), day1=str(d.index[0].date()), r1=float(-L.iloc[0]), v1=float(v.loc[d.index[0]]),
                rprev=float(r.iloc[r.index.get_loc(d.index[0]) - 1]), vprev=float(v.iloc[r.index.get_loc(d.index[0]) - 1]),
                term1=float((0.01 - (-L.iloc[0] < -v.loc[d.index[0]])) * (-L.iloc[0] + v.loc[d.index[0]])))


def c1_series():
    """The C1 portfolios (same code as c1_capital): daily simple returns in %."""
    lr_b = joint_returns(BVB, start=C_START)
    lr_b = lr_b[(lr_b != 0).any(axis=1)]
    start = str(lr_b.index[0].date())
    Rb = 100 * (np.exp(lr_b) - 1)
    Rp_b = Rb.mean(axis=1)
    Rp_s = 100 * (np.exp(joint_returns(['SPY']).loc[start:]['SPY']) - 1)
    return {'bvb': Rp_b, 'spy': Rp_s}


def fig_c1_stress(C1):
    """C1: historical ES 2.5% on rolling 250-day windows; the chosen stress windows (hatched)."""
    _style()
    ser = c1_series()
    es25 = lambda y: hs_var_es(y, 0.025)[1]   # noqa: E731
    fig, ax = plt.subplots(figsize=(W_FIG, 2.4))
    out = {}
    for k, c, lab in [('bvb', IDAred, 'BVB blue chips, equal weights'), ('spy', MainBlue, 'S&P 500 (SPY)')]:
        L = -ser[k]
        roll = pd.Series([es25(L.iloc[i - 250:i].values) for i in range(250, len(L) + 1)], index=L.index[249:])
        ax.plot(roll.index, roll.values, color=c, lw=1.0, label=f'{lab}: rolling 250-day ES 2.5%')
        a, b = pd.Timestamp(C1[k]['stress_from']), pd.Timestamp(C1[k]['stress_to'])
        ax.axvspan(a, b, color=c, alpha=0.12, lw=0, label=f'Stress window, {lab.split(",")[0].split(" (")[0]}')
        out[k] = dict(max=float(roll.max()), argmax=str(roll.idxmax().date()), last=float(roll.iloc[-1]))
    ax.set_ylabel('1-day ES 2.5% (%)')
    ax.set_title('Stress-window selection, 2017-2026')
    bottom_legend(fig, ncol=2, fontsize=9.5)
    save_fig('ch7_sem_c1_stress')
    return out


def fig_c2_horizon(B1):
    """C2: survival functions of one-day and overlapping 10-day BET losses; correct and wrong VaR 1%."""
    _style()
    r = 100 * log_returns('bet')
    L1 = (-r).values
    L10 = (-r.rolling(10).sum()).dropna().values
    v1 = np.quantile(L1, 0.99)
    v10 = np.quantile(L10, 0.99)
    fig, ax = plt.subplots(figsize=(W_FIG * 0.9, 2.4))
    for x, c, lab in [(L1, MainBlue, 'One-day log losses'), (L10, IDAred, 'Overlapping 10-day log losses')]:
        xs = np.sort(x)
        sf = 1 - np.arange(1, len(xs) + 1) / len(xs)
        m = sf > 0
        ax.semilogy(xs[m], sf[m], color=c, lw=1.1, label=lab)
    ax.axhline(0.01, color=Gray, lw=0.6, ls=':')
    marks = [(v1, MainBlue, '-', f'VaR 1%, 1 day: {v1:.2f}'), (np.sqrt(10) * v1, Forest, '--', f'sqrt(10) x VaR: {np.sqrt(10) * v1:.2f}'),
             (v10, IDAred, '-', f'VaR 1%, 10 days: {v10:.2f}'), (10 * 3.945843614767814, Purple, ':', 'AI answer: 10 x VaR = 39.46')]
    for v, c, ls, lab in marks:
        ax.axvline(v, color=c, ls=ls, lw=1.0, label=lab)
    ax.set_xlim(0, 42)
    ax.set_ylim(1e-4, 1)
    ax.set_xlabel('Loss x (%)')
    ax.set_ylabel('P(loss > x)')
    bottom_legend(fig, ncol=2, fontsize=9.5)
    save_fig('ch7_sem_c2_horizon')
    return dict(v1=v1, v10=v10, sqrt=np.sqrt(10) * v1, max10=float(L10.max()), n10=int(len(L10)),
                n_beyond10=int((L10 >= v10).sum()), es25=hs_var_es(L1, 0.025)[1], k25=int((L1 >= np.quantile(L1, 0.975)).sum()),
                k1=int((L1 >= v1).sum()))


def fig_c3_design():
    """C3: sample split (tuning / evaluation), cross-validation points with their exclusion windows, alignment t -> t + 10."""
    _style()
    r = log_returns('sp500', start='1990-01-01', end='2020-08-31')
    split = r.index[-2000]
    rng = np.random.default_rng(SEED)
    tune = r.loc[:split].iloc[:-1]
    cv = np.sort(rng.choice(np.arange(5, len(tune) - 5), 20, replace=False))
    fig, axes = plt.subplots(2, 1, figsize=(W_FIG, 2.9), gridspec_kw={'height_ratios': [1.2, 0.8]})
    ax = axes[0]
    ax.plot(tune.index, 100 * tune.values, color=MainBlue, lw=0.4, label=f'Tuning sample ({len(tune):,} days)')
    ev = r.loc[split:]
    ax.plot(ev.index, 100 * ev.values, color=IDAred, lw=0.4, label=f'Evaluation sample (last {len(ev):,} days)')
    for i in cv:
        ax.axvspan(tune.index[i - 5], tune.index[i + 5], color=Amber, alpha=0.5, lw=0)
    ax.plot([], [], color=Amber, lw=4, alpha=0.5, label='Cross-validation: 20 random test days, each with ±5 neighbouring days excluded')
    ax.set_ylabel('Log return (%)')
    ax.set_title(f'S&P 500, Jan 1990 - Aug 2020; evaluation from {split.date()}')
    ax = axes[1]
    ax.set_xlim(-1.5, 12)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.plot([-0.5, 11], [0.5, 0.5], color='black', lw=0.6)
    for t in range(0, 11):
        ax.plot([t, t], [0.46, 0.54], color='black', lw=0.6)
    ax.text(0, 0.36, '$t$', ha='center', fontsize=9, color='black')
    ax.text(1, 0.36, '$t+1$', ha='center', fontsize=9, color='black')
    ax.text(10, 0.36, '$t+10$', ha='center', fontsize=9, color='black')
    ax.annotate('', xy=(10, 0.66), xytext=(0, 0.66), arrowprops=dict(arrowstyle='->', color=Forest, lw=1.2))
    ax.text(5, 0.74, r'target: the daily return $r_{t+10}$, predicted from $x_t$', ha='center', fontsize=9, color=Forest)
    ax.plot([1, 1, 10, 10], [0.24, 0.18, 0.18, 0.24], color=Purple, lw=1.0)
    ax.text(5.5, 0.03, r'$r_{t+1} + \dots + r_{t+10}$: cumulative 10-day return, a different target', ha='center',
            fontsize=9, color=Purple)
    ax.text(-1.4, 0.92, 'Alignment of one training pair', fontsize=9, color='black')
    bottom_legend(fig, ncol=2, fontsize=9.5)
    save_fig('ch7_sem_c3_design')
    return dict(split=str(split.date()), n_tune=int(len(tune)), n_eval=int(len(ev)), first=str(r.index[0].date()))

## Run

In [ ]:
A1, A2, A5 = a1_normal(), a2_student(), a5_components()
for f in [a3_bond, a4_bonds, a6_cf]:
    print(f.__name__, f())
B1 = b1_bet_hs(); print('B1', B1)
print('B2', b2_bet_gpd())
B3 = b3_btc_fhs(); print('B3', B3)
B4 = b4_cf_vs_empirical(); print(pd.DataFrame(B4).round(2))
B5 = b5_components(); print('B5', B5)
B6 = b6_es_boot(); print(pd.DataFrame(B6).round(2))
C1 = c1_capital(); print('C1', C1)
# Seminar A1, A6, A8, B1 Extended: asymptotic intervals; B2 Extended: extremal index; B3 Extended: residual bootstrap; B7: CAViaR
SX = seminar_extras(); print(SX)
print('B2 Extended', extremal_index(-rets['bet'].values))
boot, draws = fhs_bootstrap(rets['btc'], return_draws=True); print('B3 Extended', boot)
CV = caviar_study('bet', fhs=rolling_conditional(rets['bet'], 2016))[0]; print('B7', CV)
# seminar charts
fig_bet_data(); fig_a1a2_tails(A1, A2); fig_a3_mass(); fig_a4_var_alpha(); fig_a5_shares(A5)
RP = fig_b5_shares(B5); fig_b5_rp(B5, RP); fig_a6_cf(); fig_a7_ru(); fig_a8_sim()
fig_b1_ci(B1, SX); fig_b6_ci(B6); fig_b2_ext(); fig_b4_methods(B4); fig_b3_boot(draws, B3, boot)
fig_b7_caviar(CV); fig_c1_stress(C1); fig_c2_horizon(B1); fig_c3_design()

![ch7_sem_bet_data](./ch7_sem_bet_data.png)

Output: `ch7_sem_bet_data.pdf`

![ch7_sem_a1a2_tails](./ch7_sem_a1a2_tails.png)

Output: `ch7_sem_a1a2_tails.pdf`

![ch7_sem_a3_mass](./ch7_sem_a3_mass.png)

Output: `ch7_sem_a3_mass.pdf`

![ch7_sem_a4_var_alpha](./ch7_sem_a4_var_alpha.png)

Output: `ch7_sem_a4_var_alpha.pdf`

![ch7_sem_a5_shares](./ch7_sem_a5_shares.png)

Output: `ch7_sem_a5_shares.pdf`

![ch7_sem_bet_hs](./ch7_sem_bet_hs.png)

Output: `ch7_sem_bet_hs.pdf`

![ch7_sem_b1_ci](./ch7_sem_b1_ci.png)

Output: `ch7_sem_b1_ci.pdf`

![ch7_sem_b6_ci](./ch7_sem_b6_ci.png)

Output: `ch7_sem_b6_ci.pdf`

![ch7_sem_bet_gpd](./ch7_sem_bet_gpd.png)

Output: `ch7_sem_bet_gpd.pdf`

![ch7_sem_btc_fhs](./ch7_sem_btc_fhs.png)

Output: `ch7_sem_btc_fhs.pdf`

![ch7_sem_b5_shares](./ch7_sem_b5_shares.png)

Output: `ch7_sem_b5_shares.pdf`

![ch7_sem_c2_horizon](./ch7_sem_c2_horizon.png)

Output: `ch7_sem_c2_horizon.pdf`

![ch7_sem_a6_cf](./ch7_sem_a6_cf.png)

Output: `ch7_sem_a6_cf.pdf`

![ch7_sem_a7_ru](./ch7_sem_a7_ru.png)

Output: `ch7_sem_a7_ru.pdf`

![ch7_sem_a8_sim](./ch7_sem_a8_sim.png)

Output: `ch7_sem_a8_sim.pdf`

![ch7_sem_b3_boot](./ch7_sem_b3_boot.png)

Output: `ch7_sem_b3_boot.pdf`

![ch7_sem_b4_methods](./ch7_sem_b4_methods.png)

Output: `ch7_sem_b4_methods.pdf`

![ch7_sem_b5_rp](./ch7_sem_b5_rp.png)

Output: `ch7_sem_b5_rp.pdf`

![ch7_sem_b2_clusters](./ch7_sem_b2_clusters.png)

Output: `ch7_sem_b2_clusters.pdf`

![ch7_sem_b7_caviar](./ch7_sem_b7_caviar.png)

Output: `ch7_sem_b7_caviar.pdf`

![ch7_sem_c1_stress](./ch7_sem_c1_stress.png)

Output: `ch7_sem_c1_stress.pdf`

![ch7_sem_capital](./ch7_sem_capital.png)

Output: `ch7_sem_capital.pdf`

![ch7_sem_c3_design](./ch7_sem_c3_design.png)

Output: `ch7_sem_c3_design.pdf`